This script will take the tif files and turn to parquet which will be used to predict on 

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import geopandas as gpd
import rasterio as rio
from rasterio.mask import mask
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = Path("/explore/nobackup/people/spotter5/anna_v/v2/predictors/abcfluxmodelv2")
OUT_DIR  = Path("/explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors")
os.makedirs(OUT_DIR, exist_ok=True)

STUDY_SHP = "/explore/nobackup/people/spotter5/anna_v/v2/studydomain/studydomain_3413_combined.shp"

DATASETS = [
    {"name": "ALT",            "subdir": "ALT_tif",                                     "kind": "alt_annual"},
    {"name": "ERA5",           "subdir": "ERA5",                                        "kind": "era5_month_code"},
    {"name": "SMAP_L4",        "subdir": "L4_SM_NRv11-4_40N+_soil_moisture_tif",        "kind": "YxxxxMxx"},
    {"name": "LAI_FPAR",       "subdir": "MCD15A3H_lai_fpar",                           "kind": "simple_YYYY_MM"},
    {"name": "LST",            "subdir": "LST",                                         "kind": "simple_YYYY_MM"},
    {"name": "MODIS_AllBands", "subdir": "MOD13A3_MYD13A3",                             "kind": "era5_month_code"},
    {"name": "TerraClimate",   "subdir": "TerraClimate",                                "kind": "simple_YYYY_MM"},
    {"name": "HiHydroSoil",    "subdir": "HiHydroSoil",                                 "kind": "static"},
    {"name": "MERIT_DEM_TPI",  "subdir": "MERIT_DEM_TPI",                               "kind": "static"},
]

# ============================================================
# LOAD STUDY DOMAIN (EPSG:3413)
# ============================================================

study = gpd.read_file(STUDY_SHP)
if study.crs is None or study.crs.to_epsg() != 3413:
    study = study.to_crs(3413)

# ============================================================
# HELPERS
# ============================================================

def normalize_name(s: str) -> str:
    if s is None:
        return ""
    s = re.sub(r"[^0-9a-zA-Z]+", "_", s.strip())
    return s.strip("_") or ""

def get_band_names(ds: rio.DatasetReader, dataset_name: str):
    descs = ds.descriptions
    names = []
    for i in range(1, ds.count + 1):
        d = descs[i-1] if descs and descs[i-1] else f"band{i}"
        d_norm = normalize_name(d)
        names.append(d_norm if d_norm else f"band{i}")
    return names

def extract_year_month(fname: str, kind: str):
    if kind == "alt_annual":
        m = re.match(r"ALT_(\d{4})(?:_grid)?\.tif$", fname)
        return (int(m.group(1)), None) if m else (None, None)

    if kind == "era5_month_code":
        m = re.match(r".*_(\d{4})_(\d{2})\d{10}-.*\.tif$", fname)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    if kind == "YxxxxMxx":
        m = re.match(r".*_Y(\d{4})M(\d{2})\.tif$", fname)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    if kind == "simple_YYYY_MM":
        m = re.match(r".*_(\d{4})_(\d{2})\.tif$", fname)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    return (None, None)

def clip_and_flatten(ds, study_gdf, band_names, round_decimals=3):
    # Reproject study area to match raster CRS
    if ds.crs != study_gdf.crs:
        study_ds = study_gdf.to_crs(ds.crs)
    else:
        study_ds = study_gdf

    shapes = list(study_ds.geometry)

    masked_arr, out_transform = mask(ds, shapes, crop=True, filled=False)

    data = masked_arr.data.astype("float32")
    mask_arr = np.ma.getmaskarray(masked_arr)
    data[mask_arr] = np.nan

    bands, H, W = data.shape
    rows, cols = np.arange(H), np.arange(W)
    rgrid, cgrid = np.meshgrid(rows, cols, indexing="ij")
    xs, ys = rio.transform.xy(out_transform, rgrid, cgrid)
    xs = np.array(xs, dtype="float32").ravel()
    ys = np.array(ys, dtype="float32").ravel()

    flat_bands = [np.round(data[b].ravel(), round_decimals) for b in range(bands)]

    valid = ~np.isnan(flat_bands[0])
    if not np.any(valid):
        return None

    out = {"x": xs[valid], "y": ys[valid]}
    for b_name, fb in zip(band_names, flat_bands):
        out[b_name] = fb[valid]

    return out

def dict_to_table(col_dict, year, month):
    n = len(next(iter(col_dict.values())))
    year_col  = np.full(n, year  if year  is not None else -1, dtype="int16")
    month_col = np.full(n, month if month is not None else -1, dtype="int8")
    col_dict = {**col_dict, "year": year_col, "month": month_col}
    return pa.Table.from_pydict(col_dict)

# ============================================================
# TRACK YEARS
# ============================================================

all_years = set()
year_to_months = defaultdict(set)

# ============================================================
# TIME-VARYING DATASETS
# ============================================================

def process_timevarying_dataset(ds_cfg):
    name, kind = ds_cfg["name"], ds_cfg["kind"]
    in_dir = ROOT_DIR / ds_cfg["subdir"]
    out_dataset_dir = OUT_DIR / name
    os.makedirs(out_dataset_dir, exist_ok=True)

    tifs = sorted([p for p in in_dir.glob("*.tif") if p.suffix == ".tif"])

    print(f"\n=== DATASET: {name} ({kind}) ===")

    # --- First, scan to find all years in this dataset ---
    years_in_dataset = set()
    for tif in tifs:
        y, m = extract_year_month(tif.name, kind)
        if y is not None:
            years_in_dataset.add(y)

    # --- Skip years that already have Parquet ---
    for year in sorted(years_in_dataset):
        out_path = out_dataset_dir / f"{name}_{year}.parquet"
        if out_path.exists():
            print(f"[SKIP already exists] {out_path}")
            continue

        print(f"  Processing YEAR {year}")

        tbl_list = []

        for tif in tifs:
            y, m = extract_year_month(tif.name, kind)
            if y != year:
                continue

            with rio.open(tif) as ds:
                band_names = get_band_names(ds, name)
                col_dict = clip_and_flatten(ds, study, band_names)
                if col_dict is None:
                    print(f"[SKIP no data in domain] {tif.name}")
                    continue

                if kind == "alt_annual":
                    # replicate 12 months
                    for month in range(1, 13):
                        year_to_months[y].add(month)
                        all_years.add(y)
                        tbl_list.append(dict_to_table(col_dict, y, month))
                else:
                    all_years.add(y)
                    year_to_months[y].add(m)
                    tbl_list.append(dict_to_table(col_dict, y, m))

        if tbl_list:
            final = pa.concat_tables(tbl_list)
            out_path = out_dataset_dir / f"{name}_{year}.parquet"
            pq.write_table(final, out_path, compression="snappy", use_dictionary=True)
            print(f"[WRITE] {out_path}")
        else:
            print(f"[WARN] No valid pixels for YEAR {year}")

# ============================================================
# STATIC DATASETS
# ============================================================

def process_static_dataset(ds_cfg):
    name = ds_cfg["name"]
    in_dir = ROOT_DIR / ds_cfg["subdir"]
    out_dataset_dir = OUT_DIR / name
    os.makedirs(out_dataset_dir, exist_ok=True)

    if not all_years:
        print(f"[WARN] No dynamic datasets processed yet → skipping static {name}")
        return

    tifs = sorted([p for p in in_dir.glob("*.tif")])

    print(f"\n=== STATIC DATASET: {name} ===")

    for tif in tifs:
        with rio.open(tif) as ds:
            band_names = get_band_names(ds, name)
            base_cols = clip_and_flatten(ds, study, band_names)
            if base_cols is None:
                print(f"[SKIP static outside domain] {tif.name}")
                continue

            for year in sorted(all_years):
                out_path = out_dataset_dir / f"{name}_{year}.parquet"

                if out_path.exists():
                    print(f"[SKIP static exists] {out_path}")
                    continue

                tables = []
                for month in sorted(year_to_months[year]):
                    n = len(base_cols["x"])
                    col_dict = {
                        "x": base_cols["x"],
                        "y": base_cols["y"],
                        "year": np.full(n, year, dtype="int16"),
                        "month": np.full(n, month, dtype="int8"),
                    }
                    for b in band_names:
                        col_dict[b] = base_cols[b]
                    tables.append(pa.Table.from_pydict(col_dict))

                year_table = pa.concat_tables(tables)
                pq.write_table(year_table, out_path, compression="snappy", use_dictionary=True)
                print(f"[WRITE] {out_path}")

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    # Pass 1: time-varying datasets
    for cfg in DATASETS:
        if cfg["kind"] != "static":
            process_timevarying_dataset(cfg)

    # Pass 2: static datasets
    for cfg in DATASETS:
        if cfg["kind"] == "static":
            process_static_dataset(cfg)

    print("\n[DONE] All datasets converted to yearly Parquet files.")



=== DATASET: ALT (alt_annual) ===
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_1997.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_1998.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_1999.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2000.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2001.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2002.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2003.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2004.parquet
[SKIP already exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_2005.parquet
[SKIP alrea

try 2

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from rasterio.mask import mask
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = Path("/explore/nobackup/people/spotter5/anna_v/v2/predictors/abcfluxmodelv2")
OUT_DIR  = Path("/explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors")
os.makedirs(OUT_DIR, exist_ok=True)

STUDY_SHP = "/explore/nobackup/people/spotter5/anna_v/v2/studydomain/studydomain_3413_combined.shp"

# Permafrost probability raster (static in time)
PERMAFROST_TIF = ROOT_DIR / "Permafrost_probability_Obu.tif"

# CO2 concentration CSV
CO2_CONT_CSV = "/explore/nobackup/people/spotter5/anna_v/v2/co2_cont.csv"

DATASETS = [
    {"name": "ALT",            "subdir": "ALT_tif",                                     "kind": "alt_annual"},
    {"name": "ERA5",           "subdir": "ERA5",                                        "kind": "era5_month_code"},
    {"name": "SMAP_L4",        "subdir": "L4_SM_NRv11-4_40N+_soil_moisture_tif",        "kind": "YxxxxMxx"},
    {"name": "LAI_FPAR",       "subdir": "MCD15A3H_lai_fpar",                           "kind": "simple_YYYY_MM"},
    {"name": "LST",            "subdir": "LST",                                         "kind": "simple_YYYY_MM"},
    {"name": "MODIS_AllBands", "subdir": "MOD13A3_MYD13A3",                             "kind": "era5_month_code"},
    {"name": "TerraClimate",   "subdir": "TerraClimate",                                "kind": "simple_YYYY_MM"},
    {"name": "HiHydroSoil",    "subdir": "HiHydroSoil",                                 "kind": "static"},
    {"name": "MERIT_DEM_TPI",  "subdir": "MERIT_DEM_TPI",                               "kind": "static"},
    # NEW: static SoilGrids (multi-tile, time-invariant)
    {"name": "SoilGrids",      "subdir": "SoilGrids",                                   "kind": "static"},
    # NEW: MOD44B yearly product, replicate single year across 12 months
    {"name": "MOD44B",         "subdir": "MOD44B",                                      "kind": "mod44b_annual"},
]

# ============================================================
# LOAD STUDY DOMAIN (EPSG:3413)
# ============================================================

study = gpd.read_file(STUDY_SHP)
if study.crs is None or study.crs.to_epsg() != 3413:
    study = study.to_crs(3413)

# ============================================================
# HELPERS
# ============================================================

def normalize_name(s: str) -> str:
    if s is None:
        return ""
    s = re.sub(r"[^0-9a-zA-Z]+", "_", s.strip())
    return s.strip("_") or ""

def get_band_names(ds: rio.DatasetReader, dataset_name: str):
    descs = ds.descriptions
    names = []
    for i in range(1, ds.count + 1):
        d = descs[i-1] if descs and descs[i-1] else f"band{i}"
        d_norm = normalize_name(d)
        names.append(d_norm if d_norm else f"band{i}")
    return names

def extract_year_month(fname: str, kind: str):
    if kind == "alt_annual":
        # ALT_2003_grid.tif or ALT_2003.tif
        m = re.match(r"ALT_(\d{4})(?:_grid)?\.tif$", fname)
        return (int(m.group(1)), None) if m else (None, None)

    if kind == "mod44b_annual":
        # MOD44B_continuousveg_2000.tif
        m = re.match(r"MOD44B_continuousveg_(\d{4})\.tif$", fname)
        return (int(m.group(1)), None) if m else (None, None)

    if kind == "era5_month_code":
        # ..._YYYY_MMxxxxxxxxxx-...tif
        m = re.match(r".*_(\d{4})_(\d{2})\d{10}-.*\.tif$", fname)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    if kind == "YxxxxMxx":
        # ..._YYYYYMM.tif
        m = re.match(r".*_Y(\d{4})M(\d{2})\.tif$", fname)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    if kind == "simple_YYYY_MM":
        # ..._YYYY_MM.tif
        m = re.match(r".*_(\d{4})_(\d{2})\.tif$", fname)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    return (None, None)

def clip_and_flatten(ds, study_gdf, band_names, round_decimals=3):
    # Reproject study area to match raster CRS
    if ds.crs != study_gdf.crs:
        study_ds = study_gdf.to_crs(ds.crs)
    else:
        study_ds = study_gdf

    shapes = list(study_ds.geometry)

    masked_arr, out_transform = mask(ds, shapes, crop=True, filled=False)

    data = masked_arr.data.astype("float32")
    mask_arr = np.ma.getmaskarray(masked_arr)
    data[mask_arr] = np.nan

    bands, H, W = data.shape
    rows, cols = np.arange(H), np.arange(W)
    rgrid, cgrid = np.meshgrid(rows, cols, indexing="ij")
    xs, ys = rio.transform.xy(out_transform, rgrid, cgrid)
    xs = np.array(xs, dtype="float32").ravel()
    ys = np.array(ys, dtype="float32").ravel()

    flat_bands = [np.round(data[b].ravel(), round_decimals) for b in range(bands)]

    # Use first band to define valid mask
    valid = ~np.isnan(flat_bands[0])
    if not np.any(valid):
        return None

    out = {"x": xs[valid], "y": ys[valid]}
    for b_name, fb in zip(band_names, flat_bands):
        out[b_name] = fb[valid]

    return out

def dict_to_table(col_dict, year, month):
    n = len(next(iter(col_dict.values())))
    year_col  = np.full(n, year  if year  is not None else -1, dtype="int16")
    month_col = np.full(n, month if month is not None else -1, dtype="int8")
    col_dict = {**col_dict, "year": year_col, "month": month_col}
    return pa.Table.from_pydict(col_dict)

# ============================================================
# TRACK YEARS
# ============================================================

all_years = set()
year_to_months = defaultdict(set)

# ============================================================
# TIME-VARYING DATASETS
# ============================================================

def process_timevarying_dataset(ds_cfg):
    name, kind = ds_cfg["name"], ds_cfg["kind"]
    in_dir = ROOT_DIR / ds_cfg["subdir"]
    out_dataset_dir = OUT_DIR / name
    os.makedirs(out_dataset_dir, exist_ok=True)

    tifs = sorted([p for p in in_dir.glob("*.tif") if p.suffix == ".tif"])

    print(f"\n=== DATASET: {name} ({kind}) ===")

    # --- First, scan to find all years in this dataset ---
    years_in_dataset = set()
    for tif in tifs:
        y, m = extract_year_month(tif.name, kind)
        if y is not None:
            years_in_dataset.add(y)

    # --- Process year by year, skipping existing Parquet ---
    for year in sorted(years_in_dataset):
        out_path = out_dataset_dir / f"{name}_{year}.parquet"
        if out_path.exists():
            print(f"[SKIP already exists] {out_path}")
            continue

        print(f"  Processing YEAR {year}")

        tbl_list = []

        for tif in tifs:
            y, m = extract_year_month(tif.name, kind)
            if y != year:
                continue

            with rio.open(tif) as ds:
                band_names = get_band_names(ds, name)
                col_dict = clip_and_flatten(ds, study, band_names)
                if col_dict is None:
                    print(f"[SKIP no data in domain] {tif.name}")
                    continue

                # ALT and MOD44B are yearly → replicate across 12 months
                if kind in ("alt_annual", "mod44b_annual"):
                    for month in range(1, 13):
                        year_to_months[y].add(month)
                        all_years.add(y)
                        tbl_list.append(dict_to_table(col_dict, y, month))
                else:
                    all_years.add(y)
                    year_to_months[y].add(m)
                    tbl_list.append(dict_to_table(col_dict, y, m))

        if tbl_list:
            final = pa.concat_tables(tbl_list)
            out_path = out_dataset_dir / f"{name}_{year}.parquet"
            pq.write_table(final, out_path, compression="snappy", use_dictionary=True)
            print(f"[WRITE] {out_path}")
        else:
            print(f"[WARN] No valid pixels for YEAR {year}")

# ============================================================
# STATIC DATASETS (HiHydroSoil, MERIT_DEM_TPI, SoilGrids, etc.)
# ============================================================

def process_static_dataset(ds_cfg):
    """
    Static datasets:
      - Use same value per pixel for all year/month combos.
      - If there are multiple tiles (e.g., SoilGrids), merge them
        into a single base grid by concatenating pixels.
    """
    name = ds_cfg["name"]
    in_dir = ROOT_DIR / ds_cfg["subdir"]
    out_dataset_dir = OUT_DIR / name
    os.makedirs(out_dataset_dir, exist_ok=True)

    if not all_years:
        print(f"[WARN] No dynamic datasets processed yet → skipping static {name}")
        return

    tifs = sorted([p for p in in_dir.glob("*.tif")])
    if not tifs:
        print(f"[WARN] No static tifs found in {in_dir} for {name}")
        return

    print(f"\n=== STATIC DATASET: {name} ===")

    # ---- Build combined base grid over all tiles ----
    base_cols_all = None
    band_names_all = None

    for tif in tifs:
        with rio.open(tif) as ds:
            band_names = get_band_names(ds, name)
            base_cols = clip_and_flatten(ds, study, band_names)
            if base_cols is None:
                print(f"[SKIP static outside domain] {tif.name}")
                continue

            if base_cols_all is None:
                base_cols_all = base_cols
                band_names_all = band_names
            else:
                # concatenate per key (x, y, and each band)
                for k in base_cols_all.keys():
                    base_cols_all[k] = np.concatenate(
                        [base_cols_all[k], base_cols[k]]
                    )

    if base_cols_all is None:
        print(f"[WARN] No valid pixels for static dataset {name}")
        return

    # ---- Replicate across all years/months ----
    for year in sorted(all_years):
        out_path = out_dataset_dir / f"{name}_{year}.parquet"

        if out_path.exists():
            print(f"[SKIP static exists] {out_path}")
            continue

        tables = []
        for month in sorted(year_to_months[year]):
            n = len(base_cols_all["x"])
            col_dict = {
                "x": base_cols_all["x"],
                "y": base_cols_all["y"],
                "year": np.full(n, year, dtype="int16"),
                "month": np.full(n, month, dtype="int8"),
            }
            for b in band_names_all:
                col_dict[b] = base_cols_all[b]
            tables.append(pa.Table.from_pydict(col_dict))

        year_table = pa.concat_tables(tables)
        pq.write_table(year_table, out_path, compression="snappy", use_dictionary=True)
        print(f"[WRITE] {out_path}")

# ============================================================
# PERMAFROST PROBABILITY (STATIC, SINGLE TIF)
# ============================================================

def process_permafrost_probability():
    """
    Use Permafrost_probability_Obu.tif and replicate values
    for all years and months in (all_years, year_to_months).
    """
    if not all_years:
        print("[WARN] No dynamic datasets processed yet → skipping Permafrost.")
        return

    if not PERMAFROST_TIF.exists():
        print(f"[WARN] Permafrost TIF not found: {PERMAFROST_TIF}")
        return

    name = "Permafrost_probability_Obu"
    out_dataset_dir = OUT_DIR / name
    os.makedirs(out_dataset_dir, exist_ok=True)

    print(f"\n=== STATIC DATASET: {name} ===")
    with rio.open(PERMAFROST_TIF) as ds:
        band_names = get_band_names(ds, name)
        base_cols = clip_and_flatten(ds, study, band_names)
        if base_cols is None:
            print(f"[SKIP Permafrost outside domain] {PERMAFROST_TIF.name}")
            return

        for year in sorted(all_years):
            out_path = out_dataset_dir / f"{name}_{year}.parquet"
            if out_path.exists():
                print(f"[SKIP permafrost exists] {out_path}")
                continue

            tables = []
            for month in sorted(year_to_months[year]):
                n = len(base_cols["x"])
                col_dict = {
                    "x": base_cols["x"],
                    "y": base_cols["y"],
                    "year": np.full(n, year, dtype="int16"),
                    "month": np.full(n, month, dtype="int8"),
                }
                for b in band_names:
                    col_dict[b] = base_cols[b]
                tables.append(pa.Table.from_pydict(col_dict))

            year_table = pa.concat_tables(tables)
            pq.write_table(year_table, out_path, compression="snappy", use_dictionary=True)
            print(f"[WRITE] {out_path}")

# ============================================================
# CO2 CONT (SCALAR PER YEAR/MONTH, UNIFORM FOR ALL PIXELS)
# ============================================================

def process_co2_cont():
    """
    Build CO2_CONT yearly Parquet files using:
      - spatial grid from a template raster (ALT_tif)
      - per-(year, month) scalar 'value' from CO2_CONT_CSV
    """
    if not all_years:
        print("[WARN] No dynamic datasets processed yet → skipping CO2_CONT.")
        return

    if not os.path.exists(CO2_CONT_CSV):
        print(f"[WARN] CO2 CSV not found: {CO2_CONT_CSV}")
        return

    out_dataset_dir = OUT_DIR / "CO2_CONT"
    os.makedirs(out_dataset_dir, exist_ok=True)

    print("\n=== TIME-VARYING DATASET: CO2_CONT (from CSV) ===")

    # --- Load CO2 data and aggregate to year-month ---
    co2_df = pd.read_csv(CO2_CONT_CSV)

    co2_agg = (
        co2_df
        .groupby(["year", "month"], as_index=False)["value"]
        .mean()
    )

    co2_map = {
        (int(row["year"]), int(row["month"])): float(row["value"])
        for _, row in co2_agg.iterrows()
    }

    # --- Get a template grid (x, y) from one ALT raster ---
    alt_dir = ROOT_DIR / "ALT_tif"
    alt_tifs = sorted(alt_dir.glob("ALT_*.tif"))
    if not alt_tifs:
        print(f"[WARN] No ALT tifs found in {alt_dir} → cannot build CO2_CONT grid.")
        return

    template_tif = alt_tifs[0]
    print(f"  Using template grid from: {template_tif}")

    with rio.open(template_tif) as ds:
        # band name doesn't matter; we only keep x,y
        base_cols = clip_and_flatten(ds, study, ["template"], round_decimals=0)
        if base_cols is None:
            print("[WARN] Template grid has no valid pixels in domain for CO2_CONT.")
            return

        xs = base_cols["x"].astype("float32")
        ys = base_cols["y"].astype("float32")

    n_pix = len(xs)
    print(f"  Template grid pixels: {n_pix}")

    # --- Build Parquet per year using year_to_months ---
    for year in sorted(all_years):
        out_path = out_dataset_dir / f"CO2_CONT_{year}.parquet"
        if out_path.exists():
            print(f"[SKIP CO2_CONT exists] {out_path}")
            continue

        tables = []
        for month in sorted(year_to_months[year]):
            key = (int(year), int(month))
            if key not in co2_map:
                print(f"[WARN] No CO2 value for year={year}, month={month} → skipping.")
                continue

            val = np.float32(co2_map[key])

            col_dict = {
                "x": xs,
                "y": ys,
                "year": np.full(n_pix, year, dtype="int16"),
                "month": np.full(n_pix, month, dtype="int8"),
                "co2_cont": np.full(n_pix, val, dtype="float32"),
            }
            tables.append(pa.Table.from_pydict(col_dict))

        if tables:
            year_table = pa.concat_tables(tables)
            pq.write_table(year_table, out_path, compression="snappy", use_dictionary=True)
            print(f"[WRITE] {out_path}")
        else:
            print(f"[WARN] No CO2 entries matched for YEAR {year}")

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    # Pass 1: time-varying datasets (including MOD44B yearly → months)
    for cfg in DATASETS:
        if cfg["kind"] not in ("static",):
            process_timevarying_dataset(cfg)

    # Pass 2: static datasets (HiHydroSoil, MERIT_DEM_TPI, SoilGrids)
    for cfg in DATASETS:
        if cfg["kind"] == "static":
            process_static_dataset(cfg)

    # Pass 3: Permafrost probability
    process_permafrost_probability()

    # Pass 4: CO2 concentration (co2_cont)
    process_co2_cont()

    print("\n[DONE] All datasets converted to yearly Parquet files (including Permafrost + CO2_CONT + SoilGrids + MOD44B).")



=== DATASET: ALT (alt_annual) ===
  Processing YEAR 1997


Now join all predictors to be used to apply models by year/month

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Join per-dataset predictor Parquet files into a single
per-(year, month) Parquet dataset.

Inputs (existing):
  /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/<DATASET>/<DATASET>_<YEAR>.parquet

Output (this script):
  /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined/<YEAR>/predictors_<YEAR>_<MM>.parquet

Join keys:
  x, y, year, month

Predictor columns:
  All non-key columns from each dataset, prefixed with "<DATASET>__".
"""

import os
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# ============================================================
# CONFIG
# ============================================================

IN_ROOT = Path("/explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors")
OUT_ROOT = Path("/explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_NAMES = [
    "ALT",
    "ERA5",
    "SMAP_L4",
    "LAI_FPAR",
    "LST",
    "MODIS_AllBands",
    "TerraClimate",
    "HiHydroSoil",
    "MERIT_DEM_TPI",
]

# ============================================================
# HELPERS
# ============================================================

def find_all_years():
    years = set()
    year_re = re.compile(r".*_(\d{4})\.parquet$")
    for ds_name in DATASET_NAMES:
        ds_dir = IN_ROOT / ds_name
        if not ds_dir.exists():
            continue
        for p in ds_dir.glob("*.parquet"):
            m = year_re.match(p.name)
            if m:
                years.add(int(m.group(1)))
    return sorted(years)


def load_year_tables(year):
    tables_by_ds = {}
    months_by_ds = {}

    for ds_name in DATASET_NAMES:
        ds_dir = IN_ROOT / ds_name
        if not ds_dir.exists():
            continue

        path = ds_dir / f"{ds_name}_{year}.parquet"
        if not path.exists():
            continue

        print(f"  [LOAD] {path}")
        table = pq.read_table(path)
        tables_by_ds[ds_name] = table

        if "month" in table.column_names:
            months = set(table.column("month").to_pylist())
            months = {m for m in months if m is not None and m >= 1}
        else:
            months = set()
        months_by_ds[ds_name] = months

    return tables_by_ds, months_by_ds


def table_month_to_df(table, month, ds_name):
    import pyarrow.compute as pc

    mask = pc.equal(table["month"], pa.scalar(month, type=table["month"].type))
    filtered = table.filter(mask)

    if filtered.num_rows == 0:
        return None

    df = filtered.to_pandas()
    key_cols = ["x", "y", "year", "month"]
    for k in key_cols:
        if k not in df.columns:
            raise ValueError(f"Expected key column '{k}' not found in dataset {ds_name}")

    df = df.set_index(key_cols)

    df = df.rename(columns={col: f"{ds_name}__{col}" for col in df.columns})

    return df


# ============================================================
# MAIN
# ============================================================

def main():
    years = find_all_years()
    if not years:
        print("[ERROR] No yearly parquet files found in IN_ROOT.")
        return

    print(f"Found years: {years}")

    for year in years:
        print(f"\n=== YEAR {year} ===")

        year_out_dir = OUT_ROOT / f"{year:04d}"
        year_out_dir.mkdir(parents=True, exist_ok=True)

        tables_by_ds, months_by_ds = load_year_tables(year)
        if not tables_by_ds:
            print(f"[WARN] No dataset tables found for year {year}, skipping.")
            continue

        months_all = sorted({m for ms in months_by_ds.values() for m in ms})
        if not months_all:
            print(f"[WARN] No months found in any dataset for year {year}, skipping.")
            continue

        print(f"  Months in year {year}: {months_all}")

        for month in months_all:
            out_path = year_out_dir / f"predictors_{year}_{month:02d}.parquet"

            # ============================================
            # SKIP IF OUTPUT FILE ALREADY EXISTS
            # ============================================
            if out_path.exists():
                print(f"  [SKIP exists] {out_path}")
                continue

            print(f"  [MONTH] {year}-{month:02d} → {out_path.name}")

            combined_df = None

            for ds_name, table in tables_by_ds.items():
                if month not in months_by_ds.get(ds_name, set()):
                    continue

                df_ds = table_month_to_df(table, month, ds_name)
                if df_ds is None or df_ds.empty:
                    continue

                if combined_df is None:
                    combined_df = df_ds
                else:
                    combined_df = combined_df.join(df_ds, how="outer")

            if combined_df is None or combined_df.empty:
                print(f"    [WARN] No data for {year}-{month:02d}, skipping.")
                continue

            combined_df = combined_df.reset_index()

            table_out = pa.Table.from_pandas(combined_df, preserve_index=False)
            pq.write_table(table_out, out_path, compression="snappy", use_dictionary=True)
            print(f"    [WRITE] {out_path}")

    print("\n[DONE] All years/months combined successfully.")


if __name__ == "__main__":
    main()


Found years: [1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

=== YEAR 1997 ===
  [LOAD] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors/ALT/ALT_1997.parquet
  Months in year 1997: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  [SKIP exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined/1997/predictors_1997_01.parquet
  [SKIP exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined/1997/predictors_1997_02.parquet
  [SKIP exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined/1997/predictors_1997_03.parquet
  [SKIP exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined/1997/predictors_1997_04.parquet
  [SKIP exists] /explore/nobackup/people/spotter5/anna_v/v2/parquet_predictors_joined/1997/predictors_1997_05.parquet
  [SKIP exists] /explore/nobackup/people/spotter5/ann

In [ ]:
't'